# Multiverse Analysis with Strategus in Data2Evidence

A multiverse analysis of **42 defensible analytical specifications** for a single
comparative question, on the Eunomia demo dataset (GiBleed).

**Question.** Target = diclofenac (cohort 2), comparator = celecoxib (cohort 1),
outcome = GI bleed (cohort 3). Note the direction: a risk ratio below 1 means
diclofenac is associated with *lower* GI bleed risk than celecoxib. This is
reversed relative to the usual CohortMethod vignette, and Eunomia is synthetic
(Synthea-derived), so the effect direction carries no clinical meaning.

**Specification grid.** 3 washout periods (0 / 90 / 180 days required prior
observation) crossed with three propensity-score adjustment families:

| Family | Analyses | Varied |
|---|---|---|
| Matched Cox | 1-12 | caliper {0.2, 0.0001} x maxRatio {1, 10} |
| Stratified Cox | 13-30 | strata {4, 5, 6, 7, 8, 9} |
| IPTW Cox | 31-42 | trim {0, 1%} x maxWeight {none, 10} |

Held fixed: ATT throughout, time at risk days 1-280 anchored at cohort start,
prior outcomes removed, duplicate subjects removed, first exposure only, and the
two exposure concepts excluded from the covariate set with descendants.

## 1. Environment

Import the necessary libraries

In [ ]:
library(dplyr)
library(Strategus)
library(CohortMethod)
library(glue)

## 2. Cohort definitions

ATLAS cohort definitions held in D2E: 1 celecoxib, 2 diclofenac, 3 GI bleed.

> **Open issue.** `get_cohort_definition_set(3)` currently returns HTTP 500
> ("Error while getting atlas cohort definition") while 1 and 2 succeed. Cohort 3
> is the only one using `includeDescendants`, and Eunomia ships a trimmed
> vocabulary -- if `concept_ancestor` lacks the descendants of 192671, concept
> set resolution can fail server-side. Check "Included Concepts" for that set in
> ATLAS. The outcome cohort cannot be omitted: `tco` requires `outcomeId = 3`.

In [ ]:
cohorts_set <- c(1, 2#, 3
)
cohortDefinitionSet <- rD2E::get_cohort_definition_set(cohorts_set)

cgModule <- Strategus::CohortGeneratorModule$new()

cohortDefinitionSharedResource <- cgModule$createCohortSharedResourceSpecifications(
  cohortDefinitionSet = cohortDefinitionSet
)

## 3. Target estimand and the washout axis

The exposure concepts are excluded from the covariate set (with descendants) so
the propensity model cannot use the exposure itself as a predictor.

`washoutPeriod` is the minimum required prior time in `observation_period` -- a
data-availability requirement, not a requirement of no prior exposure to the
comparator. The new-user property comes from `firstExposureOnly` and from the
cohort definitions themselves.

In [ ]:
covs2exclude <- c(1118084, 1124300)  # celecoxib, diclofenac

covSettings <- FeatureExtraction::createDefaultCovariateSettings(
  excludedCovariateConceptIds = covs2exclude,
  addDescendantsToExclude = TRUE
)

# The washout axis lives here, not on createCreateStudyPopulationArgs().
makeDataArgs <- function(washout) {
  CohortMethod::createGetDbCohortMethodDataArgs(
    covariateSettings       = covSettings,
    washoutPeriod           = washout,
    firstExposureOnly       = TRUE,
    removeDuplicateSubjects = "remove all"
  )
}

getDbCmDataArgs_wash180 <- makeDataArgs(180)
getDbCmDataArgs_wash90  <- makeDataArgs(90)
getDbCmDataArgs_wash0   <- makeDataArgs(0)

tco <- CohortMethod::createTargetComparatorOutcomes(
  targetId     = 2,   # diclofenac
  comparatorId = 1,   # celecoxib
  outcomes     = list(CohortMethod::createOutcome(outcomeId = 3))  # GI bleed
)

### Study population settings

Now a single object, shared by all 42 specifications: only time-at-risk and
outcome handling remain here.

In [ ]:
studyPopArgs <- CohortMethod::createCreateStudyPopulationArgs(
  removeSubjectsWithPriorOutcome = TRUE,
  startAnchor                    = "cohort start",
  riskWindowStart                = 1,
  endAnchor                      = "cohort start",
  riskWindowEnd                  = 280
)

## 4. Matched Cox specifications (1-12)

In [ ]:
createMatchedCoxAnalysis <- function(analysisId, caliper, maxRatio, getDbArgs,
                                     createPsArgs = CohortMethod::createCreatePsArgs(estimator = "att")) {
  description <- glue::glue("1:{maxRatio} matched Cox, caliper = {caliper}")

  CohortMethod::createCmAnalysis(
    analysisId                        = analysisId,
    description                       = description,
    getDbCohortMethodDataArgs         = getDbArgs,
    createStudyPopArgs                = studyPopArgs,
    createPsArgs                      = createPsArgs,
    matchOnPsArgs                     = CohortMethod::createMatchOnPsArgs(caliper = caliper, maxRatio = maxRatio),
    computeSharedCovariateBalanceArgs = CohortMethod::createComputeCovariateBalanceArgs(),
    computeCovariateBalanceArgs       = CohortMethod::createComputeCovariateBalanceArgs(),
    fitOutcomeModelArgs               = CohortMethod::createFitOutcomeModelArgs(modelType = "cox")
  )
}

In [ ]:
cmAnalysis1  <- createMatchedCoxAnalysis(analysisId = 1,  caliper = 0.2,    maxRatio = 1,  getDbCmDataArgs_wash180)
cmAnalysis2  <- createMatchedCoxAnalysis(analysisId = 2,  caliper = 0.2,    maxRatio = 10, getDbCmDataArgs_wash180)
cmAnalysis3  <- createMatchedCoxAnalysis(analysisId = 3,  caliper = 0.0001, maxRatio = 1,  getDbCmDataArgs_wash180)
cmAnalysis4  <- createMatchedCoxAnalysis(analysisId = 4,  caliper = 0.0001, maxRatio = 10, getDbCmDataArgs_wash180)

cmAnalysis5  <- createMatchedCoxAnalysis(analysisId = 5,  caliper = 0.2,    maxRatio = 1,  getDbCmDataArgs_wash90)
cmAnalysis6  <- createMatchedCoxAnalysis(analysisId = 6,  caliper = 0.2,    maxRatio = 10, getDbCmDataArgs_wash90)
cmAnalysis7  <- createMatchedCoxAnalysis(analysisId = 7,  caliper = 0.0001, maxRatio = 1,  getDbCmDataArgs_wash90)
cmAnalysis8  <- createMatchedCoxAnalysis(analysisId = 8,  caliper = 0.0001, maxRatio = 10, getDbCmDataArgs_wash90)

cmAnalysis9  <- createMatchedCoxAnalysis(analysisId = 9,  caliper = 0.2,    maxRatio = 1,  getDbCmDataArgs_wash0)
cmAnalysis10 <- createMatchedCoxAnalysis(analysisId = 10, caliper = 0.2,    maxRatio = 10, getDbCmDataArgs_wash0)
cmAnalysis11 <- createMatchedCoxAnalysis(analysisId = 11, caliper = 0.0001, maxRatio = 1,  getDbCmDataArgs_wash0)
cmAnalysis12 <- createMatchedCoxAnalysis(analysisId = 12, caliper = 0.0001, maxRatio = 10, getDbCmDataArgs_wash0)

## 5. Stratified Cox specifications (13-30)

In [ ]:
createStratifiedCoxAnalysis <- function(analysisId, numberOfStrata, getDbArgs,
                                        createPsArgs = CohortMethod::createCreatePsArgs(estimator = "att")) {
  description <- glue::glue("Stratified Cox, strata = {numberOfStrata}")

  CohortMethod::createCmAnalysis(
    analysisId                        = analysisId,
    description                       = description,
    getDbCohortMethodDataArgs         = getDbArgs,
    createStudyPopArgs                = studyPopArgs,
    createPsArgs                      = createPsArgs,
    stratifyByPsArgs                  = CohortMethod::createStratifyByPsArgs(numberOfStrata = numberOfStrata),
    computeSharedCovariateBalanceArgs = CohortMethod::createComputeCovariateBalanceArgs(),
    computeCovariateBalanceArgs       = CohortMethod::createComputeCovariateBalanceArgs(),
    fitOutcomeModelArgs               = CohortMethod::createFitOutcomeModelArgs(modelType = "cox", stratified = TRUE)
  )
}

In [ ]:
cmAnalysis13 <- createStratifiedCoxAnalysis(analysisId = 13, numberOfStrata = 4, getDbCmDataArgs_wash180)
cmAnalysis14 <- createStratifiedCoxAnalysis(analysisId = 14, numberOfStrata = 5, getDbCmDataArgs_wash180)
cmAnalysis15 <- createStratifiedCoxAnalysis(analysisId = 15, numberOfStrata = 6, getDbCmDataArgs_wash180)
cmAnalysis16 <- createStratifiedCoxAnalysis(analysisId = 16, numberOfStrata = 7, getDbCmDataArgs_wash180)
cmAnalysis17 <- createStratifiedCoxAnalysis(analysisId = 17, numberOfStrata = 8, getDbCmDataArgs_wash180)
cmAnalysis18 <- createStratifiedCoxAnalysis(analysisId = 18, numberOfStrata = 9, getDbCmDataArgs_wash180)

cmAnalysis19 <- createStratifiedCoxAnalysis(analysisId = 19, numberOfStrata = 4, getDbCmDataArgs_wash90)
cmAnalysis20 <- createStratifiedCoxAnalysis(analysisId = 20, numberOfStrata = 5, getDbCmDataArgs_wash90)
cmAnalysis21 <- createStratifiedCoxAnalysis(analysisId = 21, numberOfStrata = 6, getDbCmDataArgs_wash90)
cmAnalysis22 <- createStratifiedCoxAnalysis(analysisId = 22, numberOfStrata = 7, getDbCmDataArgs_wash90)
cmAnalysis23 <- createStratifiedCoxAnalysis(analysisId = 23, numberOfStrata = 8, getDbCmDataArgs_wash90)
cmAnalysis24 <- createStratifiedCoxAnalysis(analysisId = 24, numberOfStrata = 9, getDbCmDataArgs_wash90)

cmAnalysis25 <- createStratifiedCoxAnalysis(analysisId = 25, numberOfStrata = 4, getDbCmDataArgs_wash0)
cmAnalysis26 <- createStratifiedCoxAnalysis(analysisId = 26, numberOfStrata = 5, getDbCmDataArgs_wash0)
cmAnalysis27 <- createStratifiedCoxAnalysis(analysisId = 27, numberOfStrata = 6, getDbCmDataArgs_wash0)
cmAnalysis28 <- createStratifiedCoxAnalysis(analysisId = 28, numberOfStrata = 7, getDbCmDataArgs_wash0)
cmAnalysis29 <- createStratifiedCoxAnalysis(analysisId = 29, numberOfStrata = 8, getDbCmDataArgs_wash0)
cmAnalysis30 <- createStratifiedCoxAnalysis(analysisId = 30, numberOfStrata = 9, getDbCmDataArgs_wash0)

## 6. IPTW Cox specifications (31-42)

In [ ]:
createWeightedCoxAnalysis <- function(analysisId, estimator, getDbArgs,
                                      maxWeight = NULL,
                                      trimPercentile = 0) {
  description <- glue::glue(
    "IPTW Cox, estimator = {estimator}, trim = {trimPercentile}%, max weight = {ifelse(is.null(maxWeight), 'none', maxWeight)}"
  )

  trimByPsArgs <- if (trimPercentile > 0) {
    CohortMethod::createTrimByPsArgs(trimFraction = trimPercentile / 100)
  } else {
    NULL
  }

  truncateIptwArgs <- if (!is.null(maxWeight)) {
    CohortMethod::createTruncateIptwArgs(maxWeight = maxWeight)
  } else {
    NULL
  }

  CohortMethod::createCmAnalysis(
    analysisId                        = analysisId,
    description                       = description,
    getDbCohortMethodDataArgs         = getDbArgs,
    createStudyPopArgs                = studyPopArgs,
    createPsArgs                      = CohortMethod::createCreatePsArgs(estimator = estimator),
    trimByPsArgs                      = trimByPsArgs,
    truncateIptwArgs                  = truncateIptwArgs,
    computeSharedCovariateBalanceArgs = CohortMethod::createComputeCovariateBalanceArgs(),
    computeCovariateBalanceArgs       = CohortMethod::createComputeCovariateBalanceArgs(),
    fitOutcomeModelArgs               = CohortMethod::createFitOutcomeModelArgs(
      modelType          = "cox",
      inversePtWeighting = TRUE
    )
  )
}

In [ ]:
# ATT
cmAnalysis31 <- createWeightedCoxAnalysis(31, "att", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = NULL)  # neither
cmAnalysis32 <- createWeightedCoxAnalysis(32, "att", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = NULL)  # trim only
cmAnalysis33 <- createWeightedCoxAnalysis(33, "att", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = 10)    # truncate only
cmAnalysis34 <- createWeightedCoxAnalysis(34, "att", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = 10)    # both

cmAnalysis35 <- createWeightedCoxAnalysis(35, "att", getDbCmDataArgs_wash90,  trimPercentile = 0, maxWeight = NULL)  # neither
cmAnalysis36 <- createWeightedCoxAnalysis(36, "att", getDbCmDataArgs_wash90,  trimPercentile = 1, maxWeight = NULL)  # trim only
cmAnalysis37 <- createWeightedCoxAnalysis(37, "att", getDbCmDataArgs_wash90,  trimPercentile = 0, maxWeight = 10)    # truncate only
cmAnalysis38 <- createWeightedCoxAnalysis(38, "att", getDbCmDataArgs_wash90,  trimPercentile = 1, maxWeight = 10)    # both

cmAnalysis39 <- createWeightedCoxAnalysis(39, "att", getDbCmDataArgs_wash0,   trimPercentile = 0, maxWeight = NULL)  # neither
cmAnalysis40 <- createWeightedCoxAnalysis(40, "att", getDbCmDataArgs_wash0,   trimPercentile = 1, maxWeight = NULL)  # trim only
cmAnalysis41 <- createWeightedCoxAnalysis(41, "att", getDbCmDataArgs_wash0,   trimPercentile = 0, maxWeight = 10)    # truncate only
cmAnalysis42 <- createWeightedCoxAnalysis(42, "att", getDbCmDataArgs_wash0,   trimPercentile = 1, maxWeight = 10)    # both

The ATE and ATO variants below are held out of the current grid. Adding them
would change the estimand, so they belong in an exploratory rather than a
confirmatory read-out.

In [ ]:
# # ATE
# cmAnalysis43 <- createWeightedCoxAnalysis(43, "ate", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = NULL)
# cmAnalysis44 <- createWeightedCoxAnalysis(44, "ate", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = NULL)
# cmAnalysis45 <- createWeightedCoxAnalysis(45, "ate", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = 10)
# cmAnalysis46 <- createWeightedCoxAnalysis(46, "ate", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = 10)
#
# # ATO
# cmAnalysis47 <- createWeightedCoxAnalysis(47, "ato", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = NULL)
# cmAnalysis48 <- createWeightedCoxAnalysis(48, "ato", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = NULL)
# cmAnalysis49 <- createWeightedCoxAnalysis(49, "ato", getDbCmDataArgs_wash180, trimPercentile = 0, maxWeight = 10)
# cmAnalysis50 <- createWeightedCoxAnalysis(50, "ato", getDbCmDataArgs_wash180, trimPercentile = 1, maxWeight = 10)

## 7. Assemble the analysis specification

`nSpecs` controls how many analyses are submitted. Large specification payloads
have failed at the browser XHR layer; start small, then raise it once the
endpoint accepts a submission.

In [ ]:
nSpecs <- 2 

cmModule <- Strategus::CohortMethodModule$new()

cohortMethodModuleSpecifications <- cmModule$createModuleSpecifications(
  cmAnalysisList               = mget(paste0("cmAnalysis", seq_len(nSpecs))),
  targetComparatorOutcomesList = list(tco)
)

cohortGeneratorModuleSpecifications <- cgModule$createModuleSpecifications(
  generateStats = TRUE
)

analysisSpecifications <- Strategus::createEmptyAnalysisSpecificiations() |>
  Strategus::addSharedResources(cohortDefinitionSharedResource) |>
  Strategus::addModuleSpecifications(cohortGeneratorModuleSpecifications) |>
  Strategus::addModuleSpecifications(cohortMethodModuleSpecifications)

## 8. Execution

In [ ]:
d2eOptions <- create_options(
  token_study_code = "multiverseStudyEunomia",
  upload_results   = TRUE
)

rD2E::run_strategus_flow(
  analysisSpecification = analysisSpecifications,
  options               = d2eOptions
)